# Day 11: Cleaned Company Employee Dataset

This notebook loads, inspects, cleans, validates, and exports a company employee dataset. If `Company_Employee_Dataset.csv` is not present in the workspace, a small reproducible messy source is created so the assignment remains executable.

In [2]:
from pathlib import Path
import pandas as pd

source_path = Path('Day11_Messy_Company_Employee_Dataset.csv')
output_path = Path('cleaned_company_employee_dataset.csv')

if not source_path.exists():
    raise FileNotFoundError(f'Input dataset not found: {source_path.resolve()}')

raw_df = pd.read_csv(source_path)
print(f'Source used: {source_path}')
print('Initial shape:', raw_df.shape)
print('\nInitial preview:')
display(raw_df.head())

missing_before = raw_df.isnull().sum()
duplicates_before = int(raw_df.duplicated().sum())
print('\nData types before cleaning:\n', raw_df.dtypes)
print('\nMissing values before cleaning:\n', missing_before[missing_before > 0])
print('\nDuplicate rows before cleaning:', duplicates_before)
print('\nUnique categorical values before cleaning:')
for column in ['Department', 'Gender', 'Work_Mode']:
    print(f'{column}:', sorted(raw_df[column].dropna().astype(str).unique()))

clean_df = raw_df.copy()

text_columns = clean_df.select_dtypes(include=['str', 'object']).columns
for column in text_columns:
    clean_df[column] = clean_df[column].astype('string').str.strip().replace({'': pd.NA})

for column in ['Employee_Name', 'Department', 'Job_Title', 'Gender', 'City', 'Work_Mode']:
    clean_df[column] = clean_df[column].str.title()
clean_df['Job_Title'] = clean_df['Job_Title'].str.replace(r'^Hr\b', 'HR', regex=True)

numeric_columns = ['Age', 'Annual_Salary', 'Experience_Years', 'Performance_Score']
for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors='coerce')

clean_df['Joining_Date'] = pd.to_datetime(clean_df['Joining_Date'], errors='coerce')

rows_with_missing_ids = int(clean_df['Employee_ID'].isnull().sum())
clean_df = clean_df.dropna(subset=['Employee_ID'])
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
clean_df = clean_df.drop_duplicates(subset=['Employee_ID'], keep='first').reset_index(drop=True)

for column in ['Department', 'Gender', 'Job_Title', 'City', 'Work_Mode']:
    clean_df[column] = clean_df[column].fillna(clean_df[column].mode()[0])

for column in ['Age', 'Annual_Salary', 'Experience_Years', 'Performance_Score']:
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

clean_df['Joining_Date'] = clean_df['Joining_Date'].fillna(clean_df['Joining_Date'].median())
clean_df['Age'] = clean_df['Age'].round().astype('int64')
clean_df['Performance_Score'] = clean_df['Performance_Score'].round().astype('int64')
clean_df['Annual_Salary'] = clean_df['Annual_Salary'].round(2)
clean_df['Experience_Years'] = clean_df['Experience_Years'].round(1)
clean_df['Joining_Date'] = clean_df['Joining_Date'].dt.strftime('%Y-%m-%d')

clean_df.to_csv(output_path, index=False)

missing_after = clean_df.isnull().sum()
duplicates_after = int(clean_df.duplicated().sum())
print('\nCleaned preview:')
display(clean_df.head())
print('Shape before cleaning:', raw_df.shape)
print('Shape after cleaning:', clean_df.shape)
print('Rows removed for missing Employee_ID:', rows_with_missing_ids)
print('Missing cells before cleaning:', int(missing_before.sum()))
print('Missing cells after cleaning:', int(missing_after.sum()))
print('Duplicate rows before cleaning:', duplicates_before)
print('Duplicate rows after cleaning:', duplicates_after)
print('\nData types after cleaning:\n', clean_df.dtypes)
print('\nStandardized categorical values:')
for column in ['Department', 'Gender', 'Work_Mode']:
    print(f'{column}:', sorted(clean_df[column].unique()))

assert clean_df.isnull().sum().sum() == 0
assert clean_df.duplicated().sum() == 0
assert clean_df['Employee_ID'].notna().all()
assert clean_df['Employee_ID'].is_unique
assert output_path.exists()

print('\nValidation passed: no missing values, duplicate records, or missing/non-unique employee IDs remain.')
print(f'Exported cleaned dataset to: {output_path}')
print('\nCleaning summary: inspected missing values, data types, categorical values, and duplicates; stripped whitespace and standardized text; coerced numeric and date fields; removed rows without Employee_ID; removed duplicate records; used mode imputation for categorical fields and median imputation for numeric/date fields; exported the cleaned CSV.')

Source used: Day11_Messy_Company_Employee_Dataset.csv
Initial shape: (157, 12)

Initial preview:


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,NaN,Data Scientist,48.0,Other,108371.0,13.2,2017-06-13,Pune,5.0,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31.0,Female,108824.0,12.1,2021-09-15,Mumbai,2.0,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28.0,Female,119400.0,13.2,2021-02-05,Bengaluru,3.0,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30.0,Female,111847.0,NaN,2018-08-05,Jaipur,4.0,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25.0,Female,78677.0,10.9,2018-07-24,Hyderabad,4.0,Hybrid



Data types before cleaning:
 Employee_ID              str
Employee_Name            str
Department               str
Job_Title                str
Age                  float64
Gender                   str
Annual_Salary        float64
Experience_Years     float64
Joining_Date             str
City                     str
Performance_Score    float64
Work_Mode                str
dtype: object

Missing values before cleaning:
 Department           5
Age                  4
Gender               5
Annual_Salary        5
Experience_Years     3
City                 5
Performance_Score    3
Work_Mode            2
dtype: int64

Duplicate rows before cleaning: 7

Unique categorical values before cleaning:
Department: ['Customer Success', 'Data & Analytics', 'ENGINEERING', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Sales', 'engineering']
Gender: ['Female', 'MALE', 'Male', 'Other', 'female']
Work_Mode: ['Hybrid', 'Office', 'REMOTE', 'Remote', 'remote']

Cleaned preview:


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,Engineering,Data Scientist,48,Other,108371.0,13.2,2017-06-13,Pune,5,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31,Female,108824.0,12.1,2021-09-15,Mumbai,2,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28,Female,119400.0,13.2,2021-02-05,Bengaluru,3,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30,Female,111847.0,9.2,2018-08-05,Jaipur,4,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25,Female,78677.0,10.9,2018-07-24,Hyderabad,4,Hybrid


Shape before cleaning: (157, 12)
Shape after cleaning: (150, 12)
Rows removed for missing Employee_ID: 0
Missing cells before cleaning: 32
Missing cells after cleaning: 0
Duplicate rows before cleaning: 7
Duplicate rows after cleaning: 0

Data types after cleaning:
 Employee_ID           string
Employee_Name         string
Department            string
Job_Title             string
Age                    int64
Gender                string
Annual_Salary        float64
Experience_Years     float64
Joining_Date             str
City                  string
Performance_Score      int64
Work_Mode             string
dtype: object

Standardized categorical values:
Department: ['Customer Success', 'Data & Analytics', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Sales']
Gender: ['Female', 'Male', 'Other']
Work_Mode: ['Hybrid', 'Office', 'Remote']

Validation passed: no missing values, duplicate records, or missing/non-unique employee IDs remain.
Exported cleaned dataset